# Récupération du texte intégral des articles TASS sélectionnés

Ce notebook constitue l’étape qui suit l’entraînement du classifieur thématique. À partir du corpus TASS déjà filtré par le modèle, il prépare une liste propre d’articles pertinents, puis récupère automatiquement le texte intégral de chaque article directement depuis les pages TASS.

L’objectif est donc de passer d’un corpus minimal — composé principalement des titres, dates et liens — à un corpus enrichi contenant le contenu textuel complet des articles. Ce corpus pourra ensuite être utilisé pour l’analyse qualitative, l’annotation narrative, le topic modeling ou d’autres traitements textuels.


## 1. Imports

In [1]:
import json
import pandas as pd

## 2. Chargement du corpus retenu par le classifieur

Cette section charge le fichier `tass_pred_score_gt_0575.json`, qui contient les articles sélectionnés comme pertinents par le classifieur thématique. Pour chaque article, on conserve les champs essentiels : le titre, la date de publication et l’URL.

Le résultat est transformé en `DataFrame` afin de faciliter les vérifications, les fusions éventuelles et la sauvegarde dans un format propre.


In [ ]:

input_path = "tass_pred_score_gt_0575.json"

with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []
for obj in data:
    title = obj.get("title")
    published_dt = obj.get("published_dt")
    url = obj.get("url")

    if title and published_dt:
        rows.append({
            "title": title,
            "published_dt": published_dt,
            "url": url
        })

df = pd.DataFrame(rows)

df

,title,published_dt,url
0,"Волонтеры развернули штаб ""Мы вместе"" для помо...",2022-02-18T22:21:32,https://tass.ru/obschestvo/13763103
1,"Волонтер рассказал о помощи жителям ДНР, котор...",2022-02-18T22:09:51,https://tass.ru/obschestvo/13762975
2,Роспотребнадзор уделит особое внимание организ...,2022-02-18T20:56:09,https://tass.ru/obschestvo/13762457
3,Из Луганска в Россию отправился первый автобус...,2022-02-18T20:53:55,https://tass.ru/obschestvo/13762389
4,Источник: для эвакуированных из ДНР и ЛНР разв...,2022-02-18T20:53:50,https://tass.ru/obschestvo/13762407
...,...,...,...
1795,В лагерях Подмосковья отдохнут более 450 детей...,2025-11-17T10:43:43,https://tass.ru/obschestvo/25645485
1796,Число эвакуированных жителей Двуречной в Харьк...,2025-11-24T14:58:50,https://tass.ru/obschestvo/25714785
1797,В Селидове ДНР развернули ПВР для беженцев из ...,2025-11-30T12:08:01,https://tass.ru/obschestvo/25773921
1798,В проекте РКК прошли обучение свыше 500 пересе...,2025-12-05T11:25:12,https://tass.ru/obschestvo/25828613


## 3. Chargement d’un corpus complémentaire

Cette cellule charge un second fichier, `tass_data.csv`, qui contient des articles collectés ou conservés dans un autre format. Cette étape permet de comparer les deux sources et, si nécessaire, d’ajouter au corpus principal des articles qui n’auraient pas été présents dans le fichier issu du classifieur.


In [2]:
csv_path_2 = "tass_data.csv"
df2 = pd.read_csv(csv_path_2)
df2

,Unnamed: 0,Titre de l'article,Titre fr,Date,Lien
0,0,Власти ДНР организовали массовый централизован...,Les autorités de la RPD ont organisé un départ...,2022-02-18 13:10:00,https://tass.ru/mezhdunarodnaya-panorama/13756493
1,1,"В Думе заявили, что Ростовская область справит...",La Douma a déclaré que la région de Rostov se ...,2022-02-18 13:46:00,https://tass.ru/politika/13757209
2,2,Пескову неизвестно об эвакуации жителей Донбас...,Peskov ignore l'évacuation des habitants du Do...,2022-02-18 14:10:00,https://tass.ru/politika/13757849
3,3,Власти Крыма заявили о готовности принять жите...,Les autorités de Crimée ont annoncé qu'elles é...,2022-02-18 14:18:00,https://tass.ru/politika/13757407
4,4,ДНР и ЛНР объявили об эвакуации мирных жителей...,La RPD et la LPR ont annoncé l'évacuation des ...,2022-02-18 15:19:00,https://tass.ru/mezhdunarodnaya-panorama/13758699
...,...,...,...,...,...
291,291,"Более 5,1 млн беженцев прибыли в Россию с терр...","Plus de 5,1 millions de réfugiés sont arrivés ...",2023-01-09 07:56:00,https://tass.ru/obschestvo/16757749
292,292,"С февраля 2022 года в Россию прибыли более 5,2...","Depuis février 2022, plus de 5,2 millions de r...",2023-01-30 07:25:00,https://tass.ru/obschestvo/16914809
293,293,Российское гражданство с 2019 года получили бо...,"Plus de 1,4 million d'Ukrainiens ont obtenu la...",2023-02-02 07:56:00,https://tass.ru/obschestvo/16943909
294,294,За год с Украины и из Донбасса на территорию Р...,"Au cours de l'année, 5,3 millions de réfugiés ...",2023-02-20 07:06:00,https://tass.ru/obschestvo/17091725


## 4. Harmonisation des formats et ajout des articles manquants

Le second corpus n’a pas exactement les mêmes noms de colonnes que le corpus principal. Cette cellule le remet donc dans une structure commune avec trois champs : `title`, `published_dt` et `url`.

Les URL sont ensuite normalisées afin de comparer correctement les deux corpus, même en présence de différences mineures de casse, d’espaces ou de barres finales. Seules les lignes dont l’URL n’est pas déjà présente dans le corpus principal sont ajoutées.


In [12]:
# 2) приводим второй df к схеме первого
df2_std = pd.DataFrame({
    "title": df2["Titre de l'article"].fillna(df2.get("Titre fr")),  # если нет RU-заголовка, подставим FR
    "published_dt": pd.to_datetime(df2["Date"], errors="coerce"),    # дата -> datetime
    "url": df2["Lien"],
})

# если в df published_dt строкой — можно обратно в ISO (не обязательно)
# df2_std["published_dt"] = df2_std["published_dt"].dt.strftime("%Y-%m-%dT%H:%M:%S")

# 3) добавляем только те строки, которых нет в df по url
def norm_url(s):
    return (
        s.astype(str).str.strip().str.lower()
        .str.replace(r"\s+", "", regex=True)
        .str.replace(r"/+$", "", regex=True)
        .replace({"nan": pd.NA, "none": pd.NA, "": pd.NA})
    )

df["_url_norm"] = norm_url(df["url"])
df2_std["_url_norm"] = norm_url(df2_std["url"])

to_add = df2_std[df2_std["_url_norm"].notna() & ~df2_std["_url_norm"].isin(df["_url_norm"])].copy()
print("Новых строк:", len(to_add))

df = pd.concat([df.drop(columns=["_url_norm"], errors="ignore"),
                to_add.drop(columns=["_url_norm"], errors="ignore")],
               ignore_index=True)

df.head()


Новых строк: 70


,title,published_dt,url
0,"Волонтеры развернули штаб ""Мы вместе"" для помо...",2022-02-18T22:21:32,https://tass.ru/obschestvo/13763103
1,"Волонтер рассказал о помощи жителям ДНР, котор...",2022-02-18T22:09:51,https://tass.ru/obschestvo/13762975
2,Роспотребнадзор уделит особое внимание организ...,2022-02-18T20:56:09,https://tass.ru/obschestvo/13762457
3,Из Луганска в Россию отправился первый автобус...,2022-02-18T20:53:55,https://tass.ru/obschestvo/13762389
4,Источник: для эвакуированных из ДНР и ЛНР разв...,2022-02-18T20:53:50,https://tass.ru/obschestvo/13762407


## 5. Contrôle des articles non appariés

Cette cellule sert de vérification complémentaire après l’harmonisation du second corpus. Elle identifie les articles du fichier CSV qui ne correspondent pas aux URL déjà présentes dans le corpus principal et affiche leurs dates de publication.

Ce contrôle permet de comprendre quels articles ont été ajoutés, et de vérifier que les différences entre les deux sources ne proviennent pas seulement de variations de format dans les URL.


In [13]:
def norm_url(s):
    return (
        s.astype(str).str.strip().str.lower()
        .str.replace(r"\s+", "", regex=True)
        .str.replace(r"/+$", "", regex=True)
        .replace({"nan": pd.NA, "none": pd.NA, "": pd.NA})
    )

# считаем нормализованные url отдельно, не меняя df
df_url_norm = norm_url(df["url"])
df2_url_norm = norm_url(df2_std["url"])

# строки из df2_std, которые НЕ совпали бы с df
not_matched = df2_std[
    df2_url_norm.notna() &
    ~df2_url_norm.isin(df_url_norm)
].copy()

# все значения published_dt
all_dates_not_matched = not_matched["published_dt"].copy()

print("Строк, которые НЕ совпали бы:", len(not_matched))
dates_list = not_matched["published_dt"].dropna().tolist()
print(dates_list)

Строк, которые НЕ совпали бы: 70
[Timestamp('2022-02-18 13:10:00'), Timestamp('2022-02-18 14:10:00'), Timestamp('2022-02-18 15:19:00'), Timestamp('2022-02-18 16:48:00'), Timestamp('2022-02-18 17:45:00'), Timestamp('2022-02-18 17:47:00'), Timestamp('2022-02-18 18:43:00'), Timestamp('2022-02-18 18:48:00'), Timestamp('2022-02-18 20:06:00'), Timestamp('2022-02-18 20:23:00'), Timestamp('2022-02-18 21:58:00'), Timestamp('2022-02-18 23:17:00'), Timestamp('2022-02-18 23:51:00'), Timestamp('2022-02-19 00:10:00'), Timestamp('2022-02-19 00:54:00'), Timestamp('2022-02-19 00:55:00'), Timestamp('2022-02-19 05:14:00'), Timestamp('2022-02-19 06:07:00'), Timestamp('2022-02-19 07:10:00'), Timestamp('2022-02-19 08:10:00'), Timestamp('2022-02-19 08:38:00'), Timestamp('2022-02-19 09:11:00'), Timestamp('2022-02-19 09:19:00'), Timestamp('2022-02-19 09:58:00'), Timestamp('2022-02-19 11:02:00'), Timestamp('2022-02-19 11:50:00'), Timestamp('2022-02-19 16:47:00'), Timestamp('2022-02-19 17:18:00'), Timestamp('202

## 6. Sauvegarde du corpus minimal nettoyé

Après fusion et déduplication, le corpus minimal est sauvegardé dans le dossier `tass_clean`. Le fichier produit, `tass_clean.json`, contient une liste d’articles avec leurs métadonnées principales.

Cette sauvegarde crée un point de départ stable pour l’étape suivante : le scraping du texte intégral.


In [15]:
# 1) создаём папку
out_dir = "tass_clean"
os.makedirs(out_dir, exist_ok=True)

# 2) делаем копию df и приводим даты к строке (иначе json может сломаться)
df_to_save = df.copy()

if "published_dt" in df_to_save.columns:
    df_to_save["published_dt"] = df_to_save["published_dt"].astype(str)

# 3) путь к файлу
out_path = os.path.join(out_dir, "tass_clean.json")

# 4) сохраняем в JSON
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(df_to_save.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

print("Saved to:", out_path)
print("Rows saved:", len(df_to_save))


Saved to: tass_clean/tass_clean.json
Rows saved: 1870


## 7. Préparation du scraping du texte intégral

Cette section importe les bibliothèques nécessaires au scraping des pages TASS. `Playwright` permet d’ouvrir les pages dans un navigateur automatisé, tandis que `asyncio`, `random` et `tqdm` servent à gérer l’exécution asynchrone, les pauses entre les requêtes et le suivi de la progression.


In [ ]:
import json
import os
import re
import random
import asyncio
from tqdm import tqdm
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError


## 8. Fonctions d’extraction et de sauvegarde

Les fonctions suivantes organisent tout le processus d’enrichissement du corpus. Elles permettent de charger le fichier JSON d’entrée, de nettoyer les textes, d’extraire le corps des articles à partir de sélecteurs HTML, de récupérer quelques hashtags ou mots-clés, puis de sauvegarder régulièrement le résultat.

Le scraping est conçu pour être relativement robuste : il ignore les articles déjà enrichis, utilise plusieurs tentatives en cas d’échec, introduit des pauses aléatoires pour éviter des requêtes trop agressives et effectue des sauvegardes atomiques afin de limiter le risque de corruption du fichier.


In [ ]:

def nettoyer_texte(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()

def charger_articles(input_path: str):
    print(f"Chargement du fichier JSON : {input_path}")
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, dict):
        print("Format détecté : dictionnaire avec clés numériques (0, 1, 2...)")
        keys = sorted(data.keys(), key=lambda x: int(x) if str(x).isdigit() else str(x))
        return data, keys, "dict"
    elif isinstance(data, list):
        print("Format détecté : liste d'articles")
        return data, list(range(len(data))), "list"
    else:
        raise ValueError("Format JSON inattendu (ni dict ni liste).")

def out_path_from_input(input_path: str, out_filename: str = "tass_clean_with_text.json") -> str:
    os.makedirs("tass_clean", exist_ok=True)
    return os.path.join("tass_clean", out_filename)

def save_json_atomic(data, path: str):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)

async def pause_humaine(i: int):
    await asyncio.sleep(random.uniform(2.5, 6.5))
    if i > 0 and i % 25 == 0:
        await asyncio.sleep(random.uniform(15, 45))
    if i > 0 and i % 120 == 0:
        await asyncio.sleep(random.uniform(90, 180))

# ------------- EXTRACTION TEXTE (comme ton test) -------------
async def extraire_texte_tass_playwright(page) -> str:
    # 1) Sélecteur sémantique
    body = page.locator('[itemprop="articleBody"]')
    if await body.count() > 0:
        ps = await body.locator("p").all_inner_texts()
        ps = [nettoyer_texte(p) for p in ps if nettoyer_texte(p)]
        if ps:
            return "\n".join(ps).strip()

    # 2) Selecteurs alternatifs
    candidats = ["article", "div.article__text", "div.news-content", "div.content", "div.text"]
    meilleur = ""
    for sel in candidats:
        loc = page.locator(sel)
        if await loc.count() == 0:
            continue
        ps = await loc.first.locator("p").all_inner_texts()
        ps = [nettoyer_texte(p) for p in ps if nettoyer_texte(p)]
        txt = "\n".join(ps).strip()
        if len(txt) > len(meilleur):
            meilleur = txt

    # IMPORTANT: on retourne seulement si ça ressemble à un vrai texte d'article
    if len(meilleur) >= 200:
        return meilleur

    # 3) Dernier recours: on évite inner_text("body") (trop de bruit sur TASS)
    return ""

# ------------- HASHTAGS (léger, sans meta qui timeout) -------------
async def extraire_hashtags_simple(page) -> list[str]:
    """
    TASS n'a pas toujours de meta keywords. On récupère des tags via liens internes,
    en restant conservateur.
    """
    tags = set()

    # On cherche d'abord dans 'article' si présent
    containers = [page.locator("article"), page.locator("main"), page.locator("body")]
    chosen = None
    for c in containers:
        try:
            if await c.count() > 0:
                chosen = c.first
                break
        except Exception:
            pass
    if not chosen:
        return []

    # Liens qui ressemblent à des tags/thèmes/personnes/régions
    a = chosen.locator("a")
    try:
        n = await a.count()
        for i in range(min(n, 400)):  # limite pour rester léger
            href = await a.nth(i).get_attribute("href")
            txt = nettoyer_texte(await a.nth(i).inner_text())
            if not href or not txt or len(txt) > 45:
                continue
            href = href.strip()

            # heuristique conservatrice
            if any(x in href for x in ["/tag/", "/tema/", "/person/", "/region/", "/rossiya", "/ukraina", "/dnr", "/lnr"]):
                tags.add(txt)
    except Exception:
        pass

    # nettoyage minimal
    bad = {"ТАСС", "TASS", "Главная", "Общество", "Политика"}
    tags = [t for t in tags if t not in bad]
    tags = sorted(set(tags))
    return tags

# ------------- MAIN -------------
async def enrichir_tass_all(
    input_path: str = "tass_clean/tass_clean.json",
    output_filename: str = "tass_clean_with_text.json",
    headless: bool = True,
    save_every: int = 20,
    max_retries: int = 3,
    min_len: int = 200
):
    data, keys, kind = charger_articles(input_path)
    out_path = out_path_from_input(input_path, output_filename)

    total = len(keys)
    print(f"Nombre total d'articles : {total}")
    print(f"Fichier de sortie      : {out_path}")

    async with async_playwright() as p:
        print("Lancement du navigateur Chromium...")
        browser = await p.chromium.launch(headless=headless)
        context = await browser.new_context(
            viewport={"width": 1280, "height": 720},
            user_agent="Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                       "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            locale="fr-FR",
        )
        page = await context.new_page()

        ok = 0
        skipped = 0
        failed = 0

        pbar = tqdm(total=total, desc="Articles", unit="article")

        try:
            for idx, k in enumerate(keys):
                item = data[k] if kind == "dict" else data[k]
                url = item.get("url")

                if not url:
                    skipped += 1
                    pbar.update(1)
                    continue

                # skip si déjà texte valide
                existing = item.get("text")
                if isinstance(existing, str) and len(existing.strip()) >= min_len:
                    skipped += 1
                    pbar.update(1)
                    continue

                await pause_humaine(ok + failed)

                texte = ""
                hashtags = []

                last_reason = ""

                for attempt in range(1, max_retries + 1):
                    try:
                        await page.goto(url, wait_until="domcontentloaded", timeout=60000)
                        await page.wait_for_timeout(1500)

                        # точно как в твоем тесте
                        texte = await extraire_texte_tass_playwright(page)

                        # hashtags — отдельно (не обязательно)
                        hashtags = await extraire_hashtags_simple(page)

                        if texte and len(texte.strip()) >= min_len:
                            break
                        else:
                            last_reason = "Texte vide/trop court"
                            await asyncio.sleep(random.uniform(6, 12))

                    except PlaywrightTimeoutError:
                        last_reason = "Délai dépassé"
                        await asyncio.sleep(random.uniform(8, 15))
                    except Exception as e:
                        last_reason = f"Erreur: {e}"
                        await asyncio.sleep(random.uniform(8, 15))

                if texte and len(texte.strip()) >= min_len:
                    item["text"] = texte
                    item["hashtags"] = hashtags
                    item.pop("_scrape_status", None)
                    ok += 1
                else:
                    item["text"] = ""  # лучше пусто, чем мусор из body
                    item["hashtags"] = hashtags if hashtags else []
                    item["_scrape_status"] = f"ECHEC: {last_reason}"
                    failed += 1

                # autosave
                if (ok + failed) % save_every == 0:
                    save_json_atomic(data, out_path)
                    print(f"\nSauvegarde intermédiaire : ok={ok}, échecs={failed}, sautés={skipped} -> {out_path}")

                pbar.update(1)

            # save final
            save_json_atomic(data, out_path)

        finally:
            pbar.close()
            await context.close()
            await browser.close()

    print("=== TERMINÉ ===")
    print(f"Réussites : {ok}")
    print(f"Échecs    : {failed}")
    print(f"Sautés    : {skipped}")
    print(f"Fichier   : {out_path}")

# запуск:
# await enrichir_tass_all("tass_clean/tass_clean.json", "tass_clean_with_text.json", headless=True)


## 9. Lancement de l’enrichissement du corpus

Cette cellule lance le scraping du texte intégral pour tous les articles présents dans `tass_clean.json`. Pour chaque URL, le script ouvre la page TASS, tente d’extraire le texte de l’article et ajoute deux champs au corpus : `text` et `hashtags`.

Le fichier enrichi est sauvegardé progressivement sous le nom `tass_clean_with_text.json`, ce qui permet de reprendre ou de contrôler le travail même si l’exécution est interrompue.


In [51]:
await enrichir_tass_all(
    input_path="tass_clean/tass_clean.json",
    output_filename="tass_clean_with_text.json",
    headless=True,
    save_every=10,
    max_retries=3
)


Chargement du fichier JSON : tass_clean/tass_clean.json
Format détecté : liste d'articles
Nombre total d'articles : 1870
Fichier de sortie      : tass_clean/tass_clean_with_text.json
Lancement du navigateur Chromium...


Articles:   1%|▏                                     | 11/1870 [01:08<3:33:15,  6.88s/article]


Sauvegarde intermédiaire : ok=10, échecs=0, sautés=1 -> tass_clean/tass_clean_with_text.json


Articles:   1%|▍                                     | 21/1870 [02:16<3:57:18,  7.70s/article]


Sauvegarde intermédiaire : ok=20, échecs=0, sautés=1 -> tass_clean/tass_clean_with_text.json


...


Sauvegarde intermédiaire : ok=1849, échecs=1, sautés=1 -> tass_clean/tass_clean_with_text.json


Articles: 100%|███████████████████████████████████▊| 1861/1870 [4:41:47<01:07,  7.48s/article]


Sauvegarde intermédiaire : ok=1859, échecs=1, sautés=1 -> tass_clean/tass_clean_with_text.json


Articles: 100%|████████████████████████████████████| 1870/1870 [4:42:48<00:00,  9.07s/article]


=== TERMINÉ ===
Réussites : 1868
Échecs    : 1
Sautés    : 1
Fichier   : tass_clean/tass_clean_with_text.json


## 10. Chargement du corpus enrichi

Une fois le scraping terminé, cette cellule recharge le fichier contenant les textes intégraux et le transforme en `DataFrame`. Le corpus final contient désormais les métadonnées initiales ainsi que le texte complet de l’article et les hashtags éventuellement récupérés.


In [16]:

input_path = "tass_clean_with_text.json"

with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []
for obj in data:
    title = obj.get("title")
    published_dt = obj.get("published_dt")
    url = obj.get("url")

    if title and published_dt:
        rows.append({
            "title": title,
            "published_dt": published_dt,
            "url": url, 
            "text": obj.get("text", ""),
            "hashtags": obj.get("hashtags", [])
        })

df = pd.DataFrame(rows)

df 

,title,published_dt,url,text,hashtags
0,"Волонтеры развернули штаб ""Мы вместе"" для помо...",2022-02-18T22:21:32,https://tass.ru/obschestvo/13763103,"МОСКВА, 19 февраля. /ТАСС/. Волонтеры Общеросс...",[]
1,"Волонтер рассказал о помощи жителям ДНР, котор...",2022-02-18T22:09:51,https://tass.ru/obschestvo/13762975,"РОСТОВ-НА-ДОНУ, 19 февраля. /ТАСС/. Жителей ДН...","[Кризис на Украине, Россия, Ростовская область]"
2,Роспотребнадзор уделит особое внимание организ...,2022-02-18T20:56:09,https://tass.ru/obschestvo/13762457,"МОСКВА, 19 февраля. /ТАСС/. Руководитель Роспо...","[Кризис на Украине, Попова, Анна Юрьевна, Росс..."
3,Из Луганска в Россию отправился первый автобус...,2022-02-18T20:53:55,https://tass.ru/obschestvo/13762389,"ЛУГАНСК, 18 февраля. /ТАСС/. Первый автобус с ...","[Кризис на Украине, Россия, Украина]"
4,Источник: для эвакуированных из ДНР и ЛНР разв...,2022-02-18T20:53:50,https://tass.ru/obschestvo/13762407,"МАТВЕЕВ КУРГАН /Ростовская область/, 18 феврал...","[Кризис на Украине, Россия, Ростовская область]"
...,...,...,...,...,...
1865,"""10 километров мимо неразорвавшихся мин"".",2022-04-13 05:15:00,https://tass.ru/v-strane/14340553,Очередной автобус подъезжает к пункту пропуска...,"[Владимирская область, Военная операция на Укр..."
1866,В ООН дали высокую оценку действиям России по ...,2022-06-24 11:41:00,https://tass.ru/obschestvo/15024555,"МОСКВА, 24 июня. /ТАСС/. Россия достойно и на ...","[Военная операция на Украине, Россия, Украина]"
1867,Украинские беженцы потянулись домой:,2022-07-21 13:51:00,https://tass.ru/opinions/15276009,"Потоки украинцев, приезжающих в Европу и уезжа...","[Военная операция на Украине, Кризис на Украин..."
1868,"В страны Европы с 24 февраля прибыли более 7,6...",2022-10-05 19:26:00,https://tass.ru/obschestvo/15967821,"ЖЕНЕВА, 5 октября. /ТАСС/. Управление верховно...",[Украина]


## 11. Vérification des textes manquants

Cette dernière vérification compte le nombre d’articles pour lesquels le champ `text` est vide ou absent. Elle permet d’évaluer la qualité du scraping et de repérer les pages qui devront éventuellement être reprises ou contrôlées manuellement.


In [17]:
# nombre de lignes sans texte
no_text = df["text"].apply(lambda x: not isinstance(x, str) or len(x.strip()) == 0)
num_no_text = no_text.sum()
print("Nombre de lignes sans texte:", num_no_text)

Nombre de lignes sans texte: 0
